# STAR-Pipeline — LLMs vs classical baselines for ESA telemetry anomaly detection

A post-competition write-up of an end-to-end pipeline on the **ESA Anomaly Dataset (ESA-AD)**: it benchmarks **ten** approaches to the same task — flag anomalous telemetry windows and (where possible) explain them — and fine-tunes two open-source LLMs with QLoRA.

- **Code & full analysis:** <STAR-Pipeline GitHub URL>
- **Fine-tuned models (Hugging Face):** `dyrtyData/star-pipeline-qwen3-8b-advice-gguf` (text) · `dyrtyData/star-pipeline-qwen3-vl-8b-detection` (vision)

> Not a scored competition entry (the challenge closed Aug 2025) — shared as a reference solution and an honest bake-off.

## The approach

Four model families, three input modalities, all scored with the ESA-aligned **CEF0.5** (precision-weighted F-beta) and **Affinity-F1** (interval-aware) metrics:

1. **Classical:** Isolation Forest (floor) and a Telemanom-style **LSTM** autoencoder (per-channel reconstruction error + dynamic threshold).
2. **LLM (text):** Qwen3-8B fine-tuned (QLoRA) to read the numeric window as text → `ANOMALY`/`NOMINAL` + structured `DIAGNOSIS/ADVICE/ACTION`.
3. **LLM (vision, AnomSeer-style):** Qwen3-VL-8B fine-tuned to read a **rendered PNG plot** of the window.
4. **Controls:** un-fine-tuned base (zero/few-shot), a frontier model (zero/few-shot), and a **trivial always-anomaly** baseline — to answer *"did the fine-tuning actually help?"*

In [ ]:
# Inspect the attached ESA-AD dataset (per-channel pickled DataFrames + CSV metadata).
import glob, os
ROOT = "/kaggle/input"
for d in sorted(glob.glob(os.path.join(ROOT, "*"))):
    print(d)
    for sub in sorted(glob.glob(os.path.join(d, "*")))[:8]:
        print("  ", os.path.basename(sub))

In [ ]:
# Quick, runnable baseline: Isolation Forest on one channel's rolling windows.
# (The full pipeline — RevIN windowing, LSTM, LLM fine-tunes — is in the GitHub repo.)
import numpy as np, pandas as pd, glob
from sklearn.ensemble import IsolationForest

pkls = glob.glob("/kaggle/input/**/*.pkl", recursive=True)
assert pkls, "Point this at the ESA-AD channel pickles in your attached dataset."
s = pd.read_pickle(pkls[0]).select_dtypes("number").iloc[:, 0].dropna().to_numpy()

W, STRIDE = 32, 16
wins = np.stack([s[i:i+W] for i in range(0, len(s)-W, STRIDE)])
wins = (wins - wins.mean(1, keepdims=True)) / (wins.std(1, keepdims=True) + 1e-8)  # RevIN-style
flags = IsolationForest(contamination=0.25, random_state=42).fit_predict(wins) == -1
print(f"{len(wins)} windows; {flags.mean():.1%} flagged anomalous by Isolation Forest")

## Results — the bake-off (project test material)

| Approach | Precision | Recall | F1 | CEF0.5 | Affinity-F1 |
|---|---|---|---|---|---|
| Isolation Forest | 0.127 | 0.459 | 0.188 | 0.149 | — |
| **LSTM baseline (58 ch)** | **0.785** | 0.451 | **0.552** | **0.684** | **0.649** |
| LLM detection — text (Qwen3-8B) | 0.360 | 0.609 | 0.453 | 0.392 | 0.456 |
| LLM detection — vision (Qwen3-VL) | 0.769 | 0.325 | 0.457 | 0.604 | — |
| Base Qwen3-8B — zero-shot | 0 | 0 | 0 | 0 | — |
| Base Qwen3-8B — few-shot | 0.282 | 0.824 | 0.420 | 0.325 | — |
| Frontier (Claude) — zero/few-shot | ~0.3 | ~0.25 | 0.254 / 0.239 | — | — |
| **Always-anomaly (trivial)** | 0.250 | 1.000 | 0.399 | 0.294 | — |

*(Window-level on a shuffled, balanced-subsampled test set; not the official ESA-ADB event-wise protocol — see caveats in the repo.)*

## Honest findings

- **A tuned LSTM is the best detector** (F1 0.552, precision 0.785) — direct LLM detection trades precision for recall, matching the AnomLLM literature.
- **Fine-tuning still helped, defensibly:** the fine-tuned LLM is the *only* LLM-family approach that beats the trivial always-anomaly baseline (F1 0.399) with a *balanced* precision/recall. A few-shot base "matches" its F1 only by over-flagging ~80% of windows; a frontier model prompted two ways sits at chance on context-free 10-value windows.
- **The LLM's real value is advice:** 95% high-quality *when the flag is correct* — so the deployable design is the **hybrid** (cheap high-precision detector → LLM advisor).

Full methodology, all 38 engineering deviations, and a plain-language walkthrough are in the GitHub repo.